# Tutorial 20: Cell Line Context Annotation

This tutorial demonstrates how to annotate and embed **cell line context**
using embpy's `CellLineAnnotator` and `embed_description()`.

### What this tutorial covers

1. **Annotate cell lines** from 3 public databases (Cellosaurus, DepMap, Cell Model Passports)
2. **Cell line metadata** -- tissue, disease, lineage, MSI status, contamination flags
3. **Text embeddings** of cell line descriptions for context representation
4. **Visualize** cell line similarity by tissue and disease

### Data sources

| Source | Coverage | Key metadata |
|---|---|---|
| Cellosaurus | 150k+ cell lines | Species, tissue, disease, cross-refs, contamination |
| DepMap / CCLE | ~2000 cancer lines | Lineage, mutations, growth pattern |
| Cell Model Passports | ~1700 models | Cancer type, MSI, ploidy, drug response |

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import embpy.pl as epl

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.1)
%matplotlib inline

---
## 1. Annotate cell lines

Query metadata for common cancer cell lines.

In [ ]:
from embpy.resources.cellline_annotator import CellLineAnnotator

annotator = CellLineAnnotator(rate_limit_delay=0.5)

CELL_LINES = ['A549', 'HeLa', 'MCF7', 'HCT116', 'K562',
              'HEK293', 'U2OS', 'Jurkat', 'HepG2', 'PC3']

annotations = {}
for cl in CELL_LINES:
    print(f'Annotating {cl}...')
    try:
        ann = annotator.annotate(cl)
        annotations[cl] = ann
        print(f'  Tissue: {ann.get("tissue", "?")}  '
              f'Disease: {ann.get("disease", "?")}  '
              f'Lineage: {ann.get("lineage", "?")}')
    except Exception as e:
        print(f'  FAILED: {e}')

print(f'\nAnnotated {len(annotations)}/{len(CELL_LINES)} cell lines')

In [ ]:
df = pd.DataFrame([
    {
        'name': cl,
        'tissue': ann.get('tissue', ''),
        'disease': ann.get('disease', ''),
        'lineage': ann.get('lineage', ''),
        'species': ann.get('species', ''),
        'model_type': ann.get('model_type', ''),
        'msi_status': ann.get('msi_status', ''),
        'is_problematic': ann.get('is_problematic', False),
    }
    for cl, ann in annotations.items()
])
df

---
## 2. Text descriptions for embedding

In [ ]:
for cl in list(annotations.keys())[:3]:
    text = annotator.get_text_description(cl)
    print(f'\n{cl}:')
    print(f'  {text[:200]}...' if len(text) > 200 else f'  {text}')

---
## 3. Embed cell line descriptions

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device='auto')

cl_embs = {}
for cl in annotations:
    try:
        emb = embedder.embed_description(cl, model='minilm_l6_v2', entity_type='cellline')
        cl_embs[cl] = emb
        print(f'  {cl:10s}  dim={emb.shape[0]}')
    except Exception as e:
        print(f'  {cl:10s}  FAILED: {e}')

In [ ]:
embedded_cls = [cl for cl in CELL_LINES if cl in cl_embs]

adata = ad.AnnData(
    obs=pd.DataFrame({
        'cell_line': embedded_cls,
        'tissue': [annotations[cl].get('tissue', '') for cl in embedded_cls],
        'disease': [annotations[cl].get('disease', '') for cl in embedded_cls],
        'lineage': [annotations[cl].get('lineage', '') for cl in embedded_cls],
    }, index=pd.Index(embedded_cls)),
)
adata.obsm['X_cellline_text'] = np.stack([cl_embs[cl] for cl in embedded_cls]).astype(np.float32)
print(adata)

---
## 4. Visualize cell line similarity

In [ ]:
epl.plot_similarity_heatmap(
    adata=adata, obsm_key='X_cellline_text', metric='cosine',
    labels=embedded_cls,
    title='Cell line text embedding similarity',
    annot=True, fmt='.2f',
)
plt.show()

In [ ]:
epl.embedding_clustermap(
    adata, obsm_key='X_cellline_text', n_obs=None,
    title='Cell line text embeddings (clustermap)',
)
plt.show()

In [ ]:
epl.dendrogram(
    adata, obsm_key='X_cellline_text',
    metric='cosine', linkage_method='average',
    title='Cell line hierarchical clustering',
)
plt.show()

---
## 5. Annotate cell lines in an AnnData

`annotate_adata()` adds metadata columns directly to `.obs`.

In [ ]:
adata_demo = ad.AnnData(
    obs=pd.DataFrame({
        'cell_line': ['A549', 'A549', 'MCF7', 'MCF7', 'HeLa'],
        'perturbation': ['TP53', 'BRCA1', 'EGFR', 'MYC', 'KRAS'],
    }, index=pd.Index([f'cell_{i}' for i in range(5)])),
)

adata_demo = annotator.annotate_adata(adata_demo, column='cell_line')
adata_demo.obs

---
## Summary

| Method | Input | Output |
|---|---|---|
| `CellLineAnnotator.annotate(name)` | Cell line name | Full metadata dict |
| `CellLineAnnotator.annotate_adata(adata, col)` | AnnData | .obs columns + .uns annotations |
| `CellLineAnnotator.get_text_description(name)` | Cell line name | Text for embedding |
| `embedder.embed_description(name, entity_type='cellline')` | Cell line name | Embedding vector |

### Databases queried

| Source | URL | Coverage |
|---|---|---|
| Cellosaurus | api.cellosaurus.org | 150k+ cell lines |
| DepMap / CCLE | depmap.org/portal/api | ~2000 cancer lines |
| Cell Model Passports | api.cellmodelpassports.sanger.ac.uk | ~1700 models |